In [21]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

In [22]:
rng = jax.random.key(42)

In [23]:
# two gaussian blobs in 2D feature space, labelled 0 and 1, shuffled
def make_dataset(key, n_per_class=1000):
  k0, k1, kshuffle = jax.random.split(key, 3)
  # centres at +/-3 keep the blobs linearly separable even at 1000 points per class
  class0 = jax.random.normal(k0, (n_per_class, 2)) + 3.0
  class1 = jax.random.normal(k1, (n_per_class, 2)) - 3.0
  x = jnp.vstack([class0, class1])
  y = jnp.concatenate([jnp.zeros(n_per_class), jnp.ones(n_per_class)])
  # shuffle so the training loop sees the two classes interleaved
  indices = jax.random.permutation(kshuffle, jnp.arange(x.shape[0]))
  return x[indices], y[indices]

rng, subkey = jax.random.split(rng)
x, y = make_dataset(subkey)
x.shape, y.shape

((2000, 2), (2000,))

In [24]:
# weighted sum of the inputs, equation (6): z = Wx + b
def layer(w, b, x):
  return jnp.dot(x, w) + b

# step activation, equation (3): 1 if z >= threshold else 0
def step(z, threshold=0.0):
  return jnp.where(z >= threshold, 1.0, 0.0)

# forward pass, equation (2): y' = f(Wx + b)
def perceptron(w, b, x):
  return step(layer(w, b, x))

In [25]:
# error for a single point: y - y'
def loss(y_true, y_pred):
  return (y_true - y_pred).squeeze()

# perceptron update rule, equation (5): nudge parameters by the scaled error
def update_weights(w, b, xi, error, lr):
  w = w + lr * error * xi.T
  b = b + lr * error
  return w, b

In [ ]:
# a single learning step for one example: forward pass, error, update
def train_step(w, b, xi, yi, lr):
  y_pred = perceptron(w, b, xi)
  error = loss(yi, y_pred)
  return update_weights(w, b, xi, error, lr)

# one epoch = one pass through every example in the dataset
def train(w, b, x, y, lr=0.01, epochs=1):
  for epoch in range(epochs):
    for i in range(x.shape[0]):
      w, b = train_step(w, b, x[i].reshape(1, -1), y[i], lr)
  return w, b

In [ ]:
# --- everything below is plotting/animation, not part of the learning logic ---

# scatter the two classes and draw the decision boundary of equation (1)
def plot(w, b, x, y, ax, step=None, n_steps=None):
  ax.clear()
  ax.scatter(x[y == 0][:, 0], x[y == 0][:, 1], color='red', label='Class 0', edgecolors='k')
  ax.scatter(x[y == 1][:, 0], x[y == 1][:, 1], color='blue', label='Class 1', edgecolors='k')
  ax.set_xlabel('Feature $x_1$')
  ax.set_ylabel('Feature $x_2$')
  ax.grid(True)
  ax.legend(loc='upper right')

  # rearranging w1*x1 + w2*x2 + b = 0 gives x2 = -(w1*x1 + b) / w2
  slope = -(w[0, 0] / w[1, 0]).item()
  intercept = (-b[0] / w[1, 0]).item()
  ax.axline((0, intercept), slope=slope, color='green')

  ax.set_title(f'Training step {step}/{n_steps}' if step is not None else 'Decision boundary')

In [ ]:
# initialize parameters randomly, one key per parameter
rng, wkey, bkey = jax.random.split(rng, 3)
w = jax.random.normal(wkey, (2, 1))
b = jax.random.normal(bkey, (1,))

fig, ax = plt.subplots(figsize=(8, 6))
plot(w, b, x, y, ax)

In [ ]:
# same loop as train(), with a redraw every few steps to animate the boundary
def train_animated(w, b, x, y, lr=0.01, epochs=1, plot_every=10):
  fig, ax = plt.subplots(figsize=(8, 6))
  n_steps = epochs * x.shape[0]
  step_i = 0
  for epoch in range(epochs):
    for i in range(x.shape[0]):
      w, b = train_step(w, b, x[i].reshape(1, -1), y[i], lr)

      step_i += 1
      if step_i % plot_every == 0 or step_i == n_steps:
        plot(w, b, x, y, ax, step=step_i, n_steps=n_steps)
        clear_output(wait=True)
        display(fig)
        time.sleep(0.05)
  plt.close(fig)
  return w, b

w, b = train_animated(w, b, x, y)